# Single Student Prompt Experiments

This notebook runs a clean, single-student workflow for Experiments 1-4.
It is intentionally separated from the main notebook for clear thesis presentation.

In [ ]:
# 1) Configuration and Shared Utilities
import pandas as pd

from lib.experiment_utils import (
    create_client,
    load_best_attempts_df,
    select_target_student_id,
    get_student_data,
    build_strategies,
    run_experiment_rows,
    build_strategy_summary,
    save_results,
)

MODEL_ID = "gemini-2.5-flash"
RANDOM_SEED = 42
MIN_SUBMISSIONS = 5
TARGET_STUDENT_ID = None

client = create_client()
print(f"Environment ready. Using model: {MODEL_ID}")

In [ ]:
# 2) Load Data and Select Student
best_attempts_df = load_best_attempts_df()
TARGET_STUDENT_ID = select_target_student_id(
    best_attempts_df=best_attempts_df,
    target_student_id=TARGET_STUDENT_ID,
    min_submissions=MIN_SUBMISSIONS,
)
student_data = get_student_data(best_attempts_df, TARGET_STUDENT_ID)

student_scores = best_attempts_df.groupby("SubjectID")["Score"].mean()
print(f"Selected student: {TARGET_STUDENT_ID}")
print(f"Average score: {student_scores.loc[TARGET_STUDENT_ID]:.2f}")
print(f"Total submissions: {len(student_data)}")
display(student_data.head())

In [ ]:
# 3) Strategies and Experiment Runner
strategies = build_strategies(
    focus_problem_ids=list(student_data["ProblemID"].unique())
)


def run_rows(rows_df: pd.DataFrame, sleep_seconds: float = 1.0) -> pd.DataFrame:
    return run_experiment_rows(
        rows_df=rows_df,
        client=client,
        model_id=MODEL_ID,
        strategies=strategies,
        sleep_seconds=sleep_seconds,
    )


print("Experiment runner configured.")

## Experiment 1: Single Assignment Analysis

In [ ]:
struggling_assignments = student_data[student_data["Score"] < 1.0]
target_assignment = struggling_assignments.iloc[0] if not struggling_assignments.empty else student_data.iloc[0]

print(f"Analyzing Problem {target_assignment['ProblemID']} (Score: {target_assignment['Score']:.2f})")
exp1_df = run_rows(pd.DataFrame([target_assignment]), sleep_seconds=0)

display_cols = ["SubjectID", "ProblemID", "Score"] + [c for c in exp1_df.columns if c.endswith("_Output")]
display(exp1_df[display_cols])

## Experiment 2: Multiple Assignments Analysis (5 Assignments)

In [ ]:
sample_size = min(5, len(student_data))
exp2_sample = student_data.sample(n=sample_size, random_state=RANDOM_SEED)

exp2_df = run_rows(exp2_sample, sleep_seconds=1)
display(exp2_df[["SubjectID", "ProblemID", "Score"] + [c for c in exp2_df.columns if c.endswith("_Output")]])

## Experiment 3: Full History Analysis

In [ ]:
exp3_df = run_rows(student_data, sleep_seconds=1)
print("--- Experiment 3 Results ---")
display(exp3_df)

exp3_csv = f"single_student_{TARGET_STUDENT_ID}_exp3_results.csv"
save_results(exp3_df, exp3_csv)
print(f"Saved: {exp3_csv}")

## Experiment 4: Prompt Strategy Summary (Single Student)
Compute per-strategy timing and response coverage from Experiment 3 outputs.

In [ ]:
if 'exp3_df' not in locals() or exp3_df.empty:
    raise ValueError("Run Experiment 3 first.")

exp4_summary_df = build_strategy_summary(exp3_df, strategies)
display(exp4_summary_df)

exp4_csv = f"single_student_{TARGET_STUDENT_ID}_exp4_summary.csv"
save_results(exp4_summary_df, exp4_csv)
print(f"Saved: {exp4_csv}")